----------------------------------------------------------------------------------------------------

# ***with***

----------------------------------------------------------------------------------------------------

O gerenciador de contexto ( context manager ) é a forma "limpa" e moderna de lidar com recursos que precisam ser abertos e fechados obrigatoriamente. 

No Python, ele é representado pela palavra-chave ***with***.

Se o construtor (_ _init_ _) e o destrutor (_ _del_ _) são o ciclo de vida natural, o with é um contrato de segurança.

## ***1. O Problema: O Esquecimento***

Imagine abrir um arquivo para escrever dados. Se o seu programa der um erro (crash) antes de você rodar o .close(), o arquivo pode ficar corrompido ou travado pelo sistema operacional.

O destrutor _ _del_ _ nem sempre é confiável para isso, pois o Python pode demorar para limpar a memória.

## 2. ***A Solução: O bloco with***

O with garante que, não importa o que aconteça dentro do bloco (mesmo que ocorra um erro catastrófico), o recurso será fechado ao sair dele.

Exemplo Prático com Arquivos:

In [1]:
# Forma antiga e arriscada:
f = open("notas.txt", "w")
f.write("Olá!")
# Se o código travar aqui, o arquivo não fecha.
f.close()

# Forma profissional (Gerenciador de Contexto):
with open("notas.txt", "w") as f:
    f.write("Olá!")
    # O arquivo fecha AUTOMATICAMENTE aqui, mesmo se houver erro.

## ***3. Como criar o seu (Métodos _ _enter_ _ e _ _exit_ _)***

Para uma classe funcionar com o with, ela precisa implementar dois métodos especiais:

1. _ _enter_ _: O que acontece quando você entra no with. Ele geralmente retorna o objeto que você vai usar.

2. _ _exit_ _: O que acontece quando o bloco termina. É aqui que você coloca a lógica de "limpeza" (fechar conexão, deletar temporários).

In [2]:
class RoboOperador:
    def __enter__(self):
        print("Ligando robô... (__enter__)")
        return self

    def trabalhar(self):
        print("Robô está operando...")

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Desligando robô e recolhendo ferramentas... (__exit__)")
        # Os parâmetros exc_ ajudam a tratar erros que ocorreram no bloco

with RoboOperador() as robo:
    robo.trabalhar()
# Ao chegar aqui, o robô desliga sozinho.

Ligando robô... (__enter__)
Robô está operando...
Desligando robô e recolhendo ferramentas... (__exit__)


## ***4. Por que usar with em vez de _ _del_ _?***

- Determinismo: Você sabe exatamente quando a limpeza acontece (no final do recuo do código).

- Segurança: Ele é imune a exceções. Se o código "explodir" no meio do with, o _ _exit_ _ ainda será executado.

- Legibilidade: Fica claro para quem lê seu código onde o recurso está sendo usado.

# ***Resumo Técnico***

- Construtor (_ _init_ _): Prepara o objeto.

- Gerenciador (with): Garante que o uso do objeto seja seguro e temporário.

- Destrutor (_ _del_ _): É a última linha de defesa antes da memória ser liberada pelo sistema.

----------------------------------------------------------------------------------------------------

# ***Conhecimentos Avançados - A importância do 'with'***

----------------------------------------------------------------------------------------------------

O with é tão importante no Python que existe um apelido para quem o usa bem: dizemos que o código é "Pythonico". 

Ele implementa o que chamamos de 'PEP 343', transformando um processo de quatro ou cinco linhas em apenas uma, com muito mais segurança.

Vamos mergulhar nos detalhes técnicos e em casos de uso que vão além de apenas ler arquivos.

## ***1. O que acontece "Debaixo do Capô"?***

Quando você escreve um bloco with, o Python executa uma sequência lógica rigorosa. Imagine que você está entrando em um laboratório:

1. __enter__: Você abre a porta e veste o jaleco (o recurso é preparado).

2. Corpo do bloco: Você faz seus experimentos (seu código roda).

3. __exit__: Não importa se um frasco quebrou ou se tudo correu bem, você tira o jaleco e tranca a porta antes de sair.

## ***2. Suprimindo Erros com o _ _exit_ _***

Uma característica poderosa do _ _exit_ _ é que ele recebe informações sobre qualquer erro que tenha ocorrido dentro do bloco:

- exc_type: O tipo da exceção (ex: ZeroDivisionError).

- exc_val: O valor/mensagem da exceção.

- exc_tb: O traceback (onde o erro aconteceu).

Se o método _ _exit_ _ retornar True, o Python ignora o erro e continua o programa como se nada tivesse acontecido. Se retornar False (o padrão), o erro sobe e trava o programa (a menos que haja um try/except fora).

## ***3. Usando o contextlib (O jeito fácil)***

Criar uma classe inteira com _ _enter_ _ e _ _exit_ _ pode ser cansativo. O Python oferece um decorador chamado @contextmanager que permite transformar uma função geradora em um gerenciador de contexto.

In [5]:
from contextlib import contextmanager

@contextmanager
def tag_html(tag):
    print(f"<{tag}>") # Tudo antes do 'yield' é o __enter__
    try:
        yield
    finally:
        print(f"</{tag}>") # Tudo depois do 'yield' é o __exit__

with tag_html("titulo"):
    print("Olá, Mundo!")
    # O yield faz o código 'pausar' aqui, executa o print e depois volta.

<titulo>
Olá, Mundo!
</titulo>


## ***4. Casos de Uso Reais e Comuns***

Além de arquivos, o with é essencial para:

- Locks de Threading: Evita que dois processos mexam no mesmo dado ao mesmo tempo.

In [3]:
import threading

trava = threading.Lock()

with trava:
    # Apenas um processo mexe aqui por vez
    pass

- Conexões de Banco de Dados: Garante o commit (salvar) se der certo ou rollback (cancelar) se der erro.

- Testes Unitários: Verificar se uma função realmente dispara um erro.

In [ ]:
import pytest

with pytest.raises(ValueError):
    funcao_que_deve_dar_erro()

## ***5. Multiplos Gerenciadores***

Você pode abrir vários recursos no mesmo with separando-os por vírgula. 

Isso é ótimo para ler um arquivo e escrever o resultado em outro simultaneamente:

In [ ]:
with open("origem.txt") as origem, open("destino.txt", "w") as destino:

    conteudo = origem.read()
    
    destino.write(conteudo.upper())

# ***Resumo de Ouro***

O with não é apenas sobre "fechar arquivos". É sobre encapsular lógica de configuração e limpeza. 

Se você percebe que sempre faz "A" antes de um código e "B" depois, você provavelmente deveria estar usando um with.